# 02 - Evaluacion de modelos de recomendacion

Objetivo: comparar ALS, LightGCN y SASRec con las mismas metricas de ranking y sobre una muestra pequena de usuarios de test para que el analisis sea rapido y no cargue todo el dataset en memoria.

El notebook parte de las salidas generadas por Kedro. Si algun modelo no se ha ejecutado todavia, simplemente se excluye de la comparacion y se muestra en la seccion de disponibilidad.


## 1. Setup

Se usa Spark para leer parquet y hacer los calculos pesados. Solo se convierte a Pandas cuando las tablas ya son pequenas: resumen de metricas, rankings agregados y muestras de recomendaciones.


In [1]:
from pathlib import Path
import json
import math
import os

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pyspark.sql import functions as F
from pyspark.sql import Window

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

# Evita problemas habituales de Spark local en Windows.
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = os.environ.get("PYSPARK_PYTHON", "python")
os.environ["PYSPARK_DRIVER_PYTHON"] = os.environ.get("PYSPARK_DRIVER_PYTHON", "python")


In [2]:
try:
    from kedro.framework.session import KedroSession
    from kedro.framework.startup import bootstrap_project
    from pyspark.sql import SparkSession
except ImportError as exc:
    raise ImportError(
        "Ejecuta este notebook desde el entorno del proyecto, por ejemplo con `kedro jupyter lab` o `uv run kedro jupyter lab`."
    ) from exc

PROJECT_ROOT = Path.cwd().parent
metadata = bootstrap_project(PROJECT_ROOT)

with KedroSession.create(project_path=PROJECT_ROOT) as session:
    context = session.load_context()
    catalog = context.catalog

spark = SparkSession.builder.getOrCreate()

spark.conf.set("spark.sql.shuffle.partitions", "8")

print(f"Proyecto: {metadata.project_name}")
print(f"Spark: {spark.version}")

[05/07/26 11:48:52] INFO     Using                                                                  ]8;id=3486423;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/framework/project/__init__.py\__init__.py]8;;\:]8;id=3486424;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/framework/project/__init__.py#275\275]8;;\
                             '/mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/k                
                             edro/framework/project/rich_logging.yml' as logging configuration.                    

[05/07/26 11:49:05] INFO     No typed parameter requirements found, returning original   ]8;id=3486431;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/validation/parameter_validator.py\parameter_validator.py]8;;\:]8;id=3486432;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro/validation/parameter_validator.py#108\108]8;;\
                             parameters                                                                            

[05/07/26 11:49:06] INFO     Kedro is sending anonymous usage data with the sole purpose of improving ]8;id=3486439;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro_telemetry/plugin.py\plugin.py]8;;\:]8;id=3486440;file:///mnt/c/Users/ander/amazon-recsys/.venv/lib/python3.13/site-packages/kedro_telemetry/plugin.py#242\242]8;;\
                             the product. No personal data or IP addresses are stored on our side. To              
                             opt out, set the `KEDRO_DISABLE_TELEMETRY` or `DO_NOT_TRACK` environment              
                             variables, or create a `.telemetry` file in the current working                       
                             directory with the contents `consent: false`. To hide this message,                   
                             explicitly grant or deny consent. Read more at                                        
                             https://docs.kedro.org/en/stable/about/telemetry/                                     

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/07 11:49:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Proyecto: amazon-recsys
Spark: 3.5.3


## 2. Configuracion de la muestra

La comparacion se hace sobre una muestra de usuarios de test. Asi se evitan cruces grandes usuario-item y se puede iterar rapido. Si se quiere una evaluacion mas estable, subir `MAX_TEST_USERS` o ponerlo a `None`.


In [3]:
SEED = 42
MAX_TEST_USERS = 1000
K_VALUES = [10, 20]
POSITIVE_THRESHOLD = 4.0

MODEL_RECOMMENDATION_DATASETS = {
    "Popularity": "popularity_recommendations_top_k",
    "ALS": "als_recommendations_top_k",
    "LightGCN": "lightgcn_recommendations_top_k",
    "SASRec": "sasrec_recommendations_top_k",
}

METRIC_DATASETS = {
    "Popularity": "popularity_ranking_metrics",
    "ALS": "als_ranking_metrics",
    "LightGCN": "lightgcn_ranking_metrics",
    "SASRec": "sasrec_ranking_metrics",
}

print({
    "seed": SEED,
    "max_test_users": MAX_TEST_USERS,
    "k_values": K_VALUES,
    "positive_threshold": POSITIVE_THRESHOLD,
})


{'seed': 42, 'max_test_users': 1000, 'k_values': [10, 20], 'positive_threshold': 4.0}


## 3. Comprobacion de salidas disponibles

Antes de comparar, se revisa que datasets existen en el catalogo y que los archivos esperados estan materializados en `data/`. Esto permite ejecutar el notebook aunque solo haya resultados para algunos modelos.


In [4]:
def dataset_path(dataset_name: str) -> Path | None:
    try:
        dataset = catalog._get_dataset(dataset_name)
    except Exception:
        return None

    filepath = getattr(dataset, "_filepath", None)
    if filepath is None:
        filepath = getattr(dataset, "_path", None)
    if filepath is None:
        return None
    return Path(str(filepath))


def path_exists(path: Path | None) -> bool:
    if path is None:
        return False
    if path.is_absolute():
        return path.exists()
    return (PROJECT_ROOT / path).exists()

availability_rows = []
for model_name, dataset_name in MODEL_RECOMMENDATION_DATASETS.items():
    path = dataset_path(dataset_name)
    availability_rows.append({
        "model": model_name,
        "dataset": dataset_name,
        "path": str(path) if path is not None else None,
        "available": path_exists(path),
    })

availability = pd.DataFrame(availability_rows)
availability


,model,dataset,path,available
0,Popularity,popularity_recommendations_top_k,None,False
1,ALS,als_recommendations_top_k,None,False
2,LightGCN,lightgcn_recommendations_top_k,None,False
3,SASRec,sasrec_recommendations_top_k,None,False


In [5]:
available_models = availability.loc[availability["available"], "model"].tolist()
missing_models = availability.loc[~availability["available"], "model"].tolist()

print("Modelos disponibles:", available_models)
print("Modelos sin recomendaciones materializadas:", missing_models)

if not available_models:
    raise FileNotFoundError(
        "No hay recomendaciones materializadas. Ejecuta primero los pipelines de recomendacion que quieras comparar."
    )


Modelos disponibles: []
Modelos sin recomendaciones materializadas: ['Popularity', 'ALS', 'LightGCN', 'SASRec']


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:8                                                                                    │
│                                                                                                  │
│    5 print("Modelos sin recomendaciones materializadas:", missing_models)                        │
│    6                                                                                             │
│    7 if not available_models:                                                                    │
│ ❱  8 │   raise FileNotFoundError(                                                                │
│    9 │   │   "No hay recomendaciones materializadas. Ejecuta primero los pipelines de recomen    │
│   10 │   )                                                                                       │
│   11                                                                                             │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
FileNotFoundError: No hay recomendaciones materializadas. Ejecuta primero los pipelines de recomendacion que 
quieras comparar.

## 4. Ground truth de test y muestra de usuarios

Para ranking se consideran relevantes las interacciones de test con rating alto. Esto alinea la evaluacion con LightGCN/SASRec, que optimizan recomendacion top-K mas que prediccion exacta de rating.


In [ ]:
als_test = catalog.load("als_test")

relevant_test = (
    als_test
    .filter(F.col("rating") >= F.lit(POSITIVE_THRESHOLD))
    .select(F.col("user_idx").cast("int"), F.col("item_idx").cast("int"))
    .dropDuplicates(["user_idx", "item_idx"])
)

n_relevant_users = relevant_test.select("user_idx").distinct().count()
n_relevant_events = relevant_test.count()
print(f"Usuarios con test relevante: {n_relevant_users:,}")
print(f"Interacciones relevantes en test: {n_relevant_events:,}")

sample_users = relevant_test.select("user_idx").distinct().orderBy(F.rand(SEED))
if MAX_TEST_USERS is not None:
    sample_users = sample_users.limit(MAX_TEST_USERS)

sample_users = sample_users.cache()
sample_user_count = sample_users.count()
print(f"Usuarios evaluados en la muestra: {sample_user_count:,}")

sample_ground_truth = relevant_test.join(sample_users, "user_idx", "inner").cache()
print(f"Interacciones relevantes en la muestra: {sample_ground_truth.count():,}")


## 5. Carga de recomendaciones de cada modelo

Se cargan solamente las recomendaciones de los usuarios muestreados. Despues se recalculan los rankings por si algun dataset contiene mas de `K` recomendaciones o rankings no consecutivos tras el filtrado.


In [ ]:
def load_model_recommendations(model_name: str, dataset_name: str):
    recs = catalog.load(dataset_name)
    recs = (
        recs
        .select(
            F.col("user_idx").cast("int"),
            F.col("item_idx").cast("int"),
            F.col("score").cast("double"),
            F.col("rank").cast("int"),
        )
        .join(sample_users, "user_idx", "inner")
        .dropDuplicates(["user_idx", "item_idx"])
    )
    window = Window.partitionBy("user_idx").orderBy(F.col("score").desc())
    recs = recs.withColumn("rank", F.row_number().over(window))
    recs = recs.withColumn("model", F.lit(model_name))
    return recs

recommendations = {}
for model_name in available_models:
    dataset_name = MODEL_RECOMMENDATION_DATASETS[model_name]
    recs = load_model_recommendations(model_name, dataset_name).cache()
    print(f"{model_name}: {recs.count():,} recomendaciones en la muestra")
    recommendations[model_name] = recs


## 6. Metricas ranking sobre la muestra

Se calculan metricas comparables para todos los modelos:

- `Recall@K`: proporcion de items relevantes recuperados.
- `Precision@K`: proporcion de recomendaciones top-K que son relevantes.
- `NDCG@K`: calidad del orden, dando mas peso a aciertos en posiciones altas.
- `catalog_coverage@K`: porcentaje de items distintos recomendados respecto al catalogo observado en test/train.


In [ ]:
def ranking_metrics_at_k(recs, ground_truth, k: int) -> dict:
    top_k = recs.filter(F.col("rank") <= k)
    hits = top_k.join(ground_truth, ["user_idx", "item_idx"], "inner")

    relevant_per_user = ground_truth.groupBy("user_idx").agg(F.count("item_idx").alias("n_relevant"))
    hits_per_user = hits.groupBy("user_idx").agg(F.count("item_idx").alias("n_hits"))
    recs_per_user = top_k.groupBy("user_idx").agg(F.count("item_idx").alias("n_recs"))

    per_user = (
        relevant_per_user
        .join(hits_per_user, "user_idx", "left")
        .join(recs_per_user, "user_idx", "left")
        .fillna(0, subset=["n_hits", "n_recs"])
        .withColumn("recall", F.col("n_hits") / F.col("n_relevant"))
        .withColumn("precision", F.when(F.col("n_recs") > 0, F.col("n_hits") / F.col("n_recs")).otherwise(F.lit(0.0)))
    )

    dcg = hits.withColumn("gain", 1 / (F.log2(F.col("rank") + 1)))
    dcg = dcg.groupBy("user_idx").agg(F.sum("gain").alias("dcg"))

    idcg = relevant_per_user.withColumn(
        "ideal_hits",
        F.when(F.col("n_relevant") < k, F.col("n_relevant")).otherwise(F.lit(k))
    )
    idcg_rows = []
    for row in idcg.collect():
        ideal_hits = int(row["ideal_hits"])
        ideal_value = sum(1 / math.log2(rank + 1) for rank in range(1, ideal_hits + 1))
        idcg_rows.append((int(row["user_idx"]), float(ideal_value)))
    idcg_df = spark.createDataFrame(idcg_rows, schema=["user_idx", "idcg"])

    ndcg_per_user = (
        relevant_per_user
        .select("user_idx")
        .join(dcg, "user_idx", "left")
        .join(idcg_df, "user_idx", "left")
        .fillna(0, subset=["dcg", "idcg"])
        .withColumn("ndcg", F.when(F.col("idcg") > 0, F.col("dcg") / F.col("idcg")).otherwise(F.lit(0.0)))
    )

    catalog_size = als_test.select(F.col("item_idx").cast("int")).distinct().count()
    coverage = top_k.select("item_idx").distinct().count() / catalog_size if catalog_size else 0.0

    summary = per_user.agg(
        F.avg("recall").alias("recall"),
        F.avg("precision").alias("precision"),
    ).first().asDict()
    ndcg_value = ndcg_per_user.agg(F.avg("ndcg").alias("ndcg")).first()["ndcg"]

    return {
        "recall": float(summary["recall"] or 0.0),
        "precision": float(summary["precision"] or 0.0),
        "ndcg": float(ndcg_value or 0.0),
        "catalog_coverage": float(coverage),
    }

metric_rows = []
for model_name, recs in recommendations.items():
    for k in K_VALUES:
        values = ranking_metrics_at_k(recs, sample_ground_truth, k)
        metric_rows.append({"model": model_name, "k": k, **values})

metrics_df = pd.DataFrame(metric_rows).sort_values(["k", "recall"], ascending=[True, False])
metrics_df


In [ ]:
metrics_long = metrics_df.melt(
    id_vars=["model", "k"],
    value_vars=["recall", "precision", "ndcg", "catalog_coverage"],
    var_name="metric",
    value_name="value",
)

g = sns.catplot(
    data=metrics_long,
    kind="bar",
    x="model",
    y="value",
    hue="k",
    col="metric",
    col_wrap=2,
    sharey=False,
    height=4,
    aspect=1.25,
)
g.set_xticklabels(rotation=25, ha="right")
g.set_axis_labels("Modelo", "Valor")
g.fig.suptitle("Comparacion de modelos en muestra de test", y=1.03)
plt.show()


## 7. Tabla de ganador por metrica

Esta tabla resume que modelo domina cada metrica en cada valor de `K`. Es util porque un modelo puede maximizar recall y otro tener mejor cobertura o precision.


In [ ]:
winner_rows = []
for k in K_VALUES:
    subset = metrics_df[metrics_df["k"] == k]
    for metric in ["recall", "precision", "ndcg", "catalog_coverage"]:
        best = subset.sort_values(metric, ascending=False).iloc[0]
        winner_rows.append({
            "k": k,
            "metric": metric,
            "best_model": best["model"],
            "best_value": best[metric],
        })

winners_df = pd.DataFrame(winner_rows)
winners_df


## 8. Solapamiento entre modelos

Ademas de las metricas, interesa saber si los modelos recomiendan practicamente lo mismo. Se calcula el Jaccard medio entre los conjuntos top-K de cada par de modelos por usuario.


In [ ]:
def average_jaccard(model_a: str, model_b: str, k: int) -> float:
    a = recommendations[model_a].filter(F.col("rank") <= k).select(
        "user_idx", F.col("item_idx").alias("item_a")
    )
    b = recommendations[model_b].filter(F.col("rank") <= k).select(
        "user_idx", F.col("item_idx").alias("item_b")
    )

    users = sample_users.select("user_idx")
    a_count = a.groupBy("user_idx").agg(F.countDistinct("item_a").alias("a_count"))
    b_count = b.groupBy("user_idx").agg(F.countDistinct("item_b").alias("b_count"))
    inter = a.join(
        b,
        (a.user_idx == b.user_idx) & (a.item_a == b.item_b),
        "inner",
    ).select(a.user_idx.alias("user_idx"), "item_a")
    inter_count = inter.groupBy("user_idx").agg(F.countDistinct("item_a").alias("intersection"))

    per_user = (
        users
        .join(a_count, "user_idx", "left")
        .join(b_count, "user_idx", "left")
        .join(inter_count, "user_idx", "left")
        .fillna(0)
        .withColumn("union", F.col("a_count") + F.col("b_count") - F.col("intersection"))
        .withColumn("jaccard", F.when(F.col("union") > 0, F.col("intersection") / F.col("union")).otherwise(F.lit(0.0)))
    )
    return float(per_user.agg(F.avg("jaccard")).first()[0] or 0.0)

pairs = []
for i, model_a in enumerate(available_models):
    for model_b in available_models[i + 1:]:
        for k in K_VALUES:
            pairs.append({
                "model_a": model_a,
                "model_b": model_b,
                "k": k,
                "avg_jaccard": average_jaccard(model_a, model_b, k),
            })

overlap_df = pd.DataFrame(pairs)
overlap_df


In [ ]:
if not overlap_df.empty:
    plt.figure(figsize=(9, 4))
    sns.barplot(
        data=overlap_df,
        x="model_a",
        y="avg_jaccard",
        hue="model_b",
    )
    plt.title("Solapamiento medio entre recomendaciones")
    plt.xlabel("Modelo A")
    plt.ylabel("Jaccard medio")
    plt.xticks(rotation=25, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Solo hay un modelo disponible; no se puede calcular solapamiento por pares.")


## 9. Ejemplos de recomendaciones por usuario

Se muestran algunos usuarios de la muestra para revisar manualmente como cambian las recomendaciones entre modelos. Esta parte no se usa como metrica, pero ayuda a detectar resultados raros.


In [ ]:
example_users = [row["user_idx"] for row in sample_users.limit(5).collect()]
example_rows = []

for model_name, recs in recommendations.items():
    rows = (
        recs
        .filter(F.col("user_idx").isin(example_users))
        .filter(F.col("rank") <= min(K_VALUES))
        .orderBy("user_idx", "rank")
        .select("user_idx", "model", "rank", "item_idx", "score")
        .collect()
    )
    example_rows.extend([row.asDict() for row in rows])

examples_df = pd.DataFrame(example_rows)
examples_df.head(50)


## 10. Lectura rapida de resultados

Usar esta celda despues de ejecutar el notebook para dejar una conclusion corta y consistente con las tablas anteriores.


In [ ]:
if not metrics_df.empty:
    main_k = max(K_VALUES)
    subset = metrics_df[metrics_df["k"] == main_k].copy()
    best_recall = subset.sort_values("recall", ascending=False).iloc[0]
    best_ndcg = subset.sort_values("ndcg", ascending=False).iloc[0]
    best_coverage = subset.sort_values("catalog_coverage", ascending=False).iloc[0]

    print(f"Para K={main_k}:")
    print(f"- Mejor Recall: {best_recall['model']} ({best_recall['recall']:.4f})")
    print(f"- Mejor NDCG: {best_ndcg['model']} ({best_ndcg['ndcg']:.4f})")
    print(f"- Mayor cobertura: {best_coverage['model']} ({best_coverage['catalog_coverage']:.4f})")
    print("
La decision final no deberia basarse solo en una metrica: Recall/NDCG miden acierto, cobertura mide diversidad de catalogo y el solapamiento muestra si los modelos aportan recomendaciones diferentes.")
else:
    print("No hay metricas calculadas.")


## 11. Notas para repetir el experimento

- Ejecutar primero los pipelines de los modelos que se quieran comparar.
- Mantener el mismo split temporal para todos los modelos.
- Subir `MAX_TEST_USERS` cuando se quiera una estimacion mas estable.
- Mantener `POSITIVE_THRESHOLD` fijo para que la comparacion sea consistente.
